# BFF Camera Demo - Live demo notebook (Google Colab)

Boots the FastAPI backend + NiceGUI frontend inside this Colab runtime
and exposes them via an ngrok tunnel. The user opens the printed URL
in their browser to interact with the demo.

This is the v0.1 cloud path; the production deliverable is the
Windows desktop installer (`dist/BFFDemo-Setup-0.1.0.exe`).

In [ ]:
# 0. Environment setup - clone the four repos over SSH, install deps.
# The repos are private, so the clone goes through SSH.
# An SSH key that already exists in this runtime is reused as-is; nothing is
# ever deleted. Only when no key exists is one generated, and then it is
# registered with GitHub through the API (token scope: admin:public_key).
from google.colab import userdata
import os
import subprocess
from pathlib import Path

ORG = 'mahardhika'
REF = 'v0.1.0'          # same tag in all four repos
REPOS = {
    'bff-core':     f'git@github.com:{ORG}/bff-core.git',
    'bff-backend':  f'git@github.com:{ORG}/bff-backend.git',
    'bff-frontend': f'git@github.com:{ORG}/bff-frontend.git',
    'bff-ml':       f'git@github.com:{ORG}/bff-ml.git',
}

SSH_DIR = Path.home() / '.ssh'
SSH_DIR.mkdir(parents=True, exist_ok=True)
os.chmod(SSH_DIR, 0o700)

# Reuse an existing key - never remove one.
key = next((p for p in (SSH_DIR / 'id_ed25519', SSH_DIR / 'id_rsa')
            if p.exists()), None)
if key is not None:
    print('using existing key:', key)
else:
    key = SSH_DIR / 'id_ed25519'
    subprocess.run(['ssh-keygen', '-t', 'ed25519', '-N', '', '-C', 'colab-bff',
                    '-f', str(key)], check=True)
    pub = (SSH_DIR / (key.name + '.pub')).read_text().strip()
    try:
        import requests
        token = userdata.get('GITHUB_TOKEN')
        resp = requests.post(
            'https://api.github.com/user/keys',
            headers={
                'Authorization': f'Bearer {token}',
                'Accept': 'application/vnd.github+json',
            },
            json={'title': 'colab-bff', 'key': pub},
            timeout=30,
        )
        print('register key:', resp.status_code, resp.text[:200])
    except Exception as exc:  # noqa: BLE001
        print('could not register the key automatically:', exc)
        print('add this public key in GitHub -> Settings -> SSH keys:')
        print(pub)

# Trust the GitHub host key (appended, existing entries are kept).
known_hosts = SSH_DIR / 'known_hosts'
with known_hosts.open('a') as handle:
    subprocess.run(['ssh-keyscan', '-H', 'github.com'], stdout=handle, check=False)
os.chmod(known_hosts, 0o600)

ssh_env = {**os.environ, 'GIT_SSH_COMMAND': 'ssh -o StrictHostKeyChecking=accept-new'}
for name, url in REPOS.items():
    subprocess.run(
        ['git', 'clone', '--branch', REF, '--depth', '1', url, f'/content/{name}'],
        env=ssh_env,
        check=True,
    )

# Install order matters: bff-core first - bff-backend depends on it and it is
# not published on PyPI.
%pip install -e "/content/bff-core[dev]"
%pip install -e "/content/bff-backend[cuda,cpu]"
%pip install -e "/content/bff-frontend[dev]"
%pip install -e "/content/bff-ml[dev]"
%pip install pyngrok


In [ ]:
import os
from getpass import getpass

os.environ['NGROK_AUTHTOKEN'] = getpass('ngrok authtoken: ')
os.environ['SUPABASE_URL'] = os.environ.get('SUPABASE_URL', 'https://naltduhojaazpodznvry.supabase.co')
os.environ['SUPABASE_SERVICE_ROLE_KEY'] = getpass('Supabase service role: ')
os.environ['S3_ENDPOINT_URL'] = os.environ.get('S3_ENDPOINT_URL', 'https://naltduhojaazpodznvry.supabase.co/storage/v1/s3')
os.environ['S3_REGION'] = 'ap-southeast-1'
os.environ['S3_ACCESS_KEY_ID'] = getpass('S3 access: ')
os.environ['S3_SECRET_ACCESS_KEY'] = getpass('S3 secret: ')
os.environ['MODE'] = 'colab'
os.environ['INFERENCE_PROVIDER'] = 'cuda'
os.environ['BACKEND_PORT'] = '8001'
os.environ['FRONTEND_PORT'] = '3001'
os.environ['DATABASE_MODE'] = 'public'   # use 'offline' to run without Supabase


In [ ]:
# Start the BE in the background.
%cd /content/bff-backend
import subprocess, time, os
be_log = open('be.log', 'w')
be = subprocess.Popen(
    ['python', '-m', 'backend', '--host', '0.0.0.0', '--port', '8001'],
    env=os.environ.copy(), stdout=be_log, stderr=subprocess.STDOUT)
print('BE pid:', be.pid)
time.sleep(5)
import httpx
print('BE healthz:', httpx.get('http://localhost:8001/healthz').json())

In [ ]:
# Open the ngrok tunnels BEFORE starting the frontend:
# the frontend needs the public backend URL.
from pyngrok import ngrok
ngrok.set_auth_token(os.environ['NGROK_AUTHTOKEN'])
be_tunnel = ngrok.connect(8001, 'http')
fe_tunnel = ngrok.connect(3001, 'http')
print(f'\n  Backend URL:  {be_tunnel.public_url}')
print(f'  Frontend URL: {fe_tunnel.public_url}')


In [ ]:
%cd /content/bff-frontend
# Start the FE in the background, pointing at the public backend.
import subprocess, time, os
fe_log = open('fe.log', 'w')
fe = subprocess.Popen(
    ['python', '-m', 'frontend', '--host', '0.0.0.0', '--port', '3001',
     '--backend-url', be_tunnel.public_url, '--no-browser'],
    env=os.environ.copy(), stdout=fe_log, stderr=subprocess.STDOUT)
print('FE pid:', fe.pid)
time.sleep(5)
import httpx
print('FE /:', httpx.get('http://localhost:3001/').status_code)
print('\n  Open the Frontend URL in your browser. The first request takes ~10s '
      'as the tunnel hands out the certificate.')

try:
    input('\n  Press Enter to stop the demo...')
except KeyboardInterrupt:
    pass
finally:
    ngrok.kill()
    fe.terminate(); be.terminate()
    fe.wait(timeout=5); be.wait(timeout=5)
    print('stopped.')
